In [1]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/mercury147/amazonml-test/test_source2.tsv
/kaggle/input/datasets/mercury147/amazonml-test/test_source3.tsv
/kaggle/input/datasets/mercury147/amazonml-test/test_source1.tsv
/kaggle/input/datasets/mercury147/amazon-ml-hack/train_ground_truth.tsv
/kaggle/input/datasets/mercury147/amazon-ml-hack/train_source3.tsv
/kaggle/input/datasets/mercury147/amazon-ml-hack/train_source2.tsv
/kaggle/input/datasets/mercury147/amazon-ml-hack/train_source1.tsv


In [2]:
import pandas as pd

TRAIN_PATH = "/kaggle/input/datasets/mercury147/amazon-ml-hack"

s1 = pd.read_csv(
    f"{TRAIN_PATH}/train_source1.tsv",
    sep="\t",
    dtype=str
)

s2 = pd.read_csv(
    f"{TRAIN_PATH}/train_source2.tsv",
    sep="\t",
    dtype=str
)

s3 = pd.read_csv(
    f"{TRAIN_PATH}/train_source3.tsv",
    sep="\t",
    dtype=str
)

ground_truth = pd.read_csv(
    f"{TRAIN_PATH}/train_ground_truth.tsv",
    sep="\t",
    dtype=str
)

print("Source 1:", s1.shape)
print("Source 2:", s2.shape)
print("Source 3:", s3.shape)
print("Ground Truth:", ground_truth.shape)

Source 1: (2206821, 4)
Source 2: (5034616, 4)
Source 3: (5285603, 4)
Ground Truth: (2206821, 2)


In [3]:
display(s1.head())
display(s2.head())
display(s3.head())
display(ground_truth.head())

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US


,entity_id,business_name,business_address,country
0,S3-202863386,wilfordhancock.com,"Mack Rd, Haltom City, Texas",US
1,S3-859268022,International South Consultants Private Ltd,NaN,India
2,S3-22467283,LLC Moncada Léarning Center,"5780 Fawn Ct, Fort Worth, Texas",US
3,S3-671162755,Moyna's Coffee,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",US
4,S3-960981775,Pvt. EFS Print Ventures Ltd.,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...",India


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."


In [4]:
print("Source 1 columns:", s1.columns.tolist())
print("Source 2 columns:", s2.columns.tolist())
print("Source 3 columns:", s3.columns.tolist())
print("Ground truth columns:", ground_truth.columns.tolist())

Source 1 columns: ['entity_id', 'business_name', 'business_address', 'country']
Source 2 columns: ['entity_id', 'business_name', 'business_address', 'country']
Source 3 columns: ['entity_id', 'business_name', 'business_address', 'country']
Ground truth columns: ['source1_entity_id', 'matched_entity_ids']


In [5]:
for name, df in {
    "Source 1": s1,
    "Source 2": s2,
    "Source 3": s3
}.items():

    print(f"\n{'='*50}")
    print(name)

    print("Shape:", df.shape)

    print("\nMissing values:")
    print(df.isna().sum())

    print("\nDuplicate rows:", df.duplicated().sum())

    print("Duplicate entity IDs:",
          df["entity_id"].duplicated().sum())

    print("\nCountry distribution:")
    print(df["country"].value_counts(dropna=False))


Source 1
Shape: (2206821, 4)

Missing values:
entity_id           0
business_name       0
business_address    0
country             0
dtype: int64

Duplicate rows: 0
Duplicate entity IDs: 0

Country distribution:
country
US       1323633
India     883188
Name: count, dtype: int64

Source 2
Shape: (5034616, 4)

Missing values:
entity_id                0
business_name            2
business_address    168967
country                  0
dtype: int64

Duplicate rows: 0
Duplicate entity IDs: 0

Country distribution:
country
US       3016817
India    2017799
Name: count, dtype: int64

Source 3
Shape: (5285603, 4)

Missing values:
entity_id                0
business_name           13
business_address    175916
country                  0
dtype: int64

Duplicate rows: 0
Duplicate entity IDs: 0

Country distribution:
country
US       3170056
India    2115547
Name: count, dtype: int64


In [6]:
# Normalising data
import pandas as pd
import re
import unicodedata

def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Unicode canonical normalization
    text = unicodedata.normalize("NFC", text)

    # Lowercase
    text = text.lower()

    # & -> and
    text = text.replace("&", " and ")

    # Keep letters, numbers, combining marks, and whitespace
    cleaned = []

    for char in text:
        category = unicodedata.category(char)

        if (
            category.startswith(("L", "N", "M"))
            or char.isspace()
        ):
            cleaned.append(char)
        else:
            cleaned.append(" ")

    text = "".join(cleaned)

    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [7]:
#testing for -- hindi data

test_values = [
    "राम मार्केटिंग प्राइवेट लिमिटेड",
    "आदित्य प्रॉपर्टीज एलएलपी",
    "International South Consultants Private Ltd",
    "B+ Retail Inc"
]

for x in test_values:
    print(x)
    print("→", normalize_text(x))
    print()

राम मार्केटिंग प्राइवेट लिमिटेड
→ राम मार्केटिंग प्राइवेट लिमिटेड

आदित्य प्रॉपर्टीज एलएलपी
→ आदित्य प्रॉपर्टीज एलएलपी

International South Consultants Private Ltd
→ international south consultants private ltd

B+ Retail Inc
→ b retail inc



In [8]:
#normalised columns for s1

s1["name_norm"] = s1["business_name"].map(normalize_text)
s1["address_norm"] = s1["business_address"].map(normalize_text)
s1["country_norm"] = s1["country"].str.strip().str.lower()

In [9]:
s2["name_norm"] = s2["business_name"].map(normalize_text)
s2["address_norm"] = s2["business_address"].map(normalize_text)
s2["country_norm"] = s2["country"].str.strip().str.lower()

In [10]:
s3["name_norm"] = s3["business_name"].map(normalize_text)
s3["address_norm"] = s3["business_address"].map(normalize_text)
s3["country_norm"] = s3["country"].str.strip().str.lower()



In [11]:
display(
    s1[[
        "business_name",
        "name_norm",
        "business_address",
        "address_norm"
    ]].head(10)
)

,business_name,name_norm,business_address,address_norm
0,Orelee's Barbershop,orelee s barbershop,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc
1,Prime Money,prime money,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok
2,B+ Retail Inc,b retail inc,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az
3,Christ Chapel,christ chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md
4,Prabhav Business Center,prabhav business center,"797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal
5,Custom Wealth Services LLC,custom wealth services llc,"OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue
6,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",2505 tower 1 oakwood runwal greens mulund gore...
7,Nexus Anchor Rain,nexus anchor rain,"1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn
8,Moore Bitwise Inc,moore bitwise inc,"337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in
9,Dermatology Green Medicine,dermatology green medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",294 meadowcreek drive unit unit 2 village of p...


Now we'll make a second version of the business name where common legal suffixes are removed.

In [12]:
#removin common legal/biznus suffix

LEGAL_SUFFIXES = {
    "limited",
    "ltd",
    "private",
    "pvt",
    "corporation",
    "corp",
    "incorporated",
    "inc",
    "llc",
    "llp",
}

def get_name_core(name):
    if not name:
        return ""

    tokens = name.split()

    # Remove legal suffixes
    tokens = [
        token for token in tokens
        if token not in LEGAL_SUFFIXES
    ]

    return " ".join(tokens)


#and apply:
s1["name_core"] = s1["name_norm"].map(get_name_core)
s2["name_core"] = s2["name_norm"].map(get_name_core)
s3["name_core"] = s3["name_norm"].map(get_name_core)

In [13]:
#checking it:
display(
    s1[
        ["business_name", "name_norm", "name_core"]
    ].head(20)
)

,business_name,name_norm,name_core
0,Orelee's Barbershop,orelee s barbershop,orelee s barbershop
1,Prime Money,prime money,prime money
2,B+ Retail Inc,b retail inc,b retail
3,Christ Chapel,christ chapel,christ chapel
4,Prabhav Business Center,prabhav business center,prabhav business center
5,Custom Wealth Services LLC,custom wealth services llc,custom wealth services
6,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited,consulting nyasa nursing
7,Nexus Anchor Rain,nexus anchor rain,nexus anchor rain
8,Moore Bitwise Inc,moore bitwise inc,moore bitwise
9,Dermatology Green Medicine,dermatology green medicine,dermatology green medicine


now, address normalisation

In [14]:
# Address normalization

ADDRESS_ABBREVIATIONS = {
    r"\brd\b": "road",
    r"\bave\b": "avenue",
    r"\bav\b": "avenue",
    r"\bblvd\b": "boulevard",
    r"\bhwy\b": "highway",
    r"\bln\b": "lane",
    r"\bdr\b": "drive",
    r"\btrl\b": "trail",
    r"\bter\b": "terrace",
    r"\bcir\b": "circle",
    r"\bapt\b": "apartment",
    r"\bste\b": "suite",
}

def normalize_address(text):
    text = normalize_text(text)

    if not text:
        return ""

    for pattern, replacement in ADDRESS_ABBREVIATIONS.items():
        text = re.sub(pattern, replacement, text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

In [15]:
#test for ct to not turn into court

address_test = "66 Edgewood Street, Bridgeport, CT"

print(address_test)
print("→", normalize_address(address_test))

66 Edgewood Street, Bridgeport, CT
→ 66 edgewood street bridgeport ct


In [16]:
#applying to the 3 sources
s1["address_norm"] = s1["business_address"].map(normalize_address)
s2["address_norm"] = s2["business_address"].map(normalize_address)
s3["address_norm"] = s3["business_address"].map(normalize_address)

In [17]:
#and check:
display(
    s1[
        ["business_address", "address_norm"]
    ].head(20)
)

,business_address,address_norm
0,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc
1,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok
2,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az
3,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md
4,"797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal
5,"OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue
6,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",2505 tower 1 oakwood runwal greens mulund gore...
7,"1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn
8,"337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in
9,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",294 meadowcreek drive unit unit 2 village of p...


In [18]:
#extracting numbers from addresses
def extract_numbers(text):
    if not text:
        return ""

    numbers = re.findall(r"\d+", text)

    return " ".join(numbers)

#apply to al
s1["address_numbers"] = s1["address_norm"].map(extract_numbers)
s2["address_numbers"] = s2["address_norm"].map(extract_numbers)
s3["address_numbers"] = s3["address_norm"].map(extract_numbers)

In [19]:
#check it 
display(
    s1[
        ["business_address", "address_norm", "address_numbers"]
    ].head(20)
)

,business_address,address_norm,address_numbers
0,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc,1795
1,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok,17560
2,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az,1712
3,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md,2100
4,"797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal,797
5,"OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue,5559
6,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",2505 tower 1 oakwood runwal greens mulund gore...,2505 1
7,"1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn,1111 2007
8,"337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in,337
9,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",294 meadowcreek drive unit unit 2 village of p...,294 2


In [20]:
#EXTractINg POSTAL CODES


''''def extract_postal_codes(text):
    if not text:
        return ""

    # 4–6 digit numeric postal codes
    codes = re.findall(r"\b\d{4,6}\b", text)

    return " ".join(codes)

#apply
s1["postal_codes"] = s1["address_norm"].map(extract_postal_codes)
s2["postal_codes"] = s2["address_norm"].map(extract_postal_codes)
s3["postal_codes"] = s3["address_norm"].map(extract_postal_codes)'''

#not using postal codes for now, bcz wo address number ko hi postal code maan rha hai n we dont need allat redundant shi

# i dropped the postalcode  column

'''s1.drop(columns=["postal_codes"], inplace=True)
s2.drop(columns=["postal_codes"], inplace=True)
s3.drop(columns=["postal_codes"], inplace=True)'''

<>:9: SyntaxWarning: invalid escape sequence '\d'
<>:9: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_16/464775423.py:9: SyntaxWarning: invalid escape sequence '\d'
  codes = re.findall(r"\b\d{4,6}\b", text)


's1.drop(columns=["postal_codes"], inplace=True)\ns2.drop(columns=["postal_codes"], inplace=True)\ns3.drop(columns=["postal_codes"], inplace=True)'

In [21]:
#creating an address token rep bcs we dont need python lists for millions of rows 

def address_tokens(text):
    if not text:
        return ""

    return " ".join(text.split())

#apply
s1["address_tokens"] = s1["address_norm"].map(address_tokens)
s2["address_tokens"] = s2["address_norm"].map(address_tokens)
s3["address_tokens"] = s3["address_norm"].map(address_tokens)

In [22]:
#CHECKING EVERYTHING;

display(
    s1[
        [
            "business_address",
            "address_norm",
            "address_numbers",
            "address_tokens"
        ]
    ].head(20)
)

,business_address,address_norm,address_numbers,address_tokens
0,"1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc,1795,1795 westchester drive high point nc
1,"17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok,17560,17560 ellis road tahlequah ok
2,"1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az,1712,1712 montebello avenue phoenix az
3,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md,2100,2100 cameron drive unit apartment g dundalk md
4,"797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal,797,797 lake town block a kolkata howrah west bengal
5,"OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue,5559,oh columbus 5559 orville avenue
6,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",2505 tower 1 oakwood runwal greens mulund gore...,2505 1,2505 tower 1 oakwood runwal greens mulund gore...
7,"1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn,1111 2007,1111 church street unit 2007 nashville tn
8,"337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in,337,337 oakland avenue michigan city in
9,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",294 meadowcreek drive unit unit 2 village of p...,294 2,294 meadowcreek drive unit unit 2 village of p...


In [23]:
#CHECKIN FOR WHWER RECORDS HAVE MISSING DATA

display(
    s2[
        s2["business_address"].isna()
    ][
        [
            "entity_id",
            "business_name",
            "business_address",
            "address_norm",
            "address_numbers"
        ]
    ].head(10)
)

,entity_id,business_name,business_address,address_norm,address_numbers
53,S2-187379771,"Guerra And Krueger Table, LLC",NaN,,
69,S2-751730544,L E Pierce Cal [PC],NaN,,
93,S2-162849450,Preferred Bay Montage Inc,NaN,,
195,S2-616836723,DELONG STRATEGIC READY,NaN,,
236,S2-39617527,Desert Vanguard ([Maintenance]),NaN,,
346,S2-582984116,Wildlife Sóciety,NaN,,
352,S2-762023797,Orellana Kensington,NaN,,
376,S2-682660453,rathi holdings,NaN,,
390,S2-752363850,Berry Saul,NaN,,
424,S2-14102665,"Thrasher, Wanner 5tenson LLC Center",NaN,,


In [24]:
ground_truth.head(10)

,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762..."
6,S1-318373630,"S2-660036492,S3-804600254"
7,S1-86989137,"S3-274817120,S3-312496301"
8,S1-29845983,"S2-648035184,S3-588502663"
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439"


In [25]:
for df in [s1, s2, s3]:
    df["name_norm"] = df["business_name"].map(normalize_text)
    df["name_core"] = df["name_norm"].map(get_name_core)

    df["address_norm"] = df["business_address"].map(normalize_address)
    df["address_numbers"] = df["address_norm"].map(extract_numbers)

    df["country_norm"] = df["country"].str.strip().str.lower()